# SMA Gap Momentum Strategy - Analysis and Visualization

This notebook analyzes the SMA Gap Momentum strategy with comprehensive visualizations.

**Strategy Overview:**
- Analyzes gap between SMA5 and SMA20
- Classifies market signals based on gap momentum
- Predicts next-day returns
- Detects anomalies where predictions fail

In [4]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import os

# Add parent directory to path to allow importing strategies
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import strategy module
from strategies.strategy_sma_gap_momentum import (
    calculate_features,
    detect_anomalies,
    predict_returns,
    get_strategy_summary
)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Configuration

In [5]:
# Configuration
STOCK_NAME = 's1'  # Change this to analyze different stocks (s1 to s30)
DATA_DIR = '../sample_data'

print(f"Analyzing Stock: {STOCK_NAME}")
print(f"Data Directory: {DATA_DIR}")

Analyzing Stock: s1
Data Directory: ../sample_data


## 2. Load Data

In [6]:
# Load stock data
filepath = f"{DATA_DIR}/{STOCK_NAME}.npy"
A = np.load(filepath, allow_pickle=True)

# Extract columns
dates = A[:, 0]
P = A[:, 2]  # Close prices
V = A[:, 6]  # Volume

print(f"Loaded {len(P)} trading days")
print(f"Price range: {P.min():.2f} - {P.max():.2f}")
print(f"Volume range: {V.min():.0f} - {V.max():.0f}")

Loaded 242 trading days
Price range: 47.83 - 72.84
Volume range: 20 - 319


## 3. Calculate Features and Signals

In [ ]:
# Calculate technical features
df = calculate_features(P, V)

# Add dates to dataframe
df['date'] = pd.to_datetime(dates.astype(int).astype(str), format='%Y%m%d', errors='coerce')

# Display first few rows
print("Feature DataFrame (first 10 rows):")
print(df[['date', 'price', 'volume', 'Price_Direction', 'SMA5', 'SMA20', 'SMA_Diff', 'Strategy_Signal']].head(10))

## 4. Detect Anomalies

In [ ]:
# Detect anomalies
df = detect_anomalies(df)

# Get summary statistics
summary = get_strategy_summary(df)

print("="*70)
print(f"STRATEGY SUMMARY: {STOCK_NAME}")
print("="*70)
print(f"\nTotal Trading Days: {summary['total_days']}")
print(f"\nSignal Distribution:")
print(f"  Strong Downtrend:  {summary['strong_downtrend_days']:3d} days")
print(f"  Signal Uptrend:    {summary['signal_uptrend_days']:3d} days")
print(f"  Strong Uptrend:    {summary['strong_uptrend_days']:3d} days")
print(f"  Signal Downtrend:  {summary['signal_downtrend_days']:3d} days")
print(f"\nAnomaly Counts:")
print(f"  Strong Downtrend Anomalies: {summary['down_anomalies']:3d}")
print(f"  Signal Downtrend Anomalies: {summary['signaldown_anomalies']:3d}")
print(f"  Signal Uptrend Anomalies:   {summary['signalup_anomalies']:3d}")
print(f"  Strong Uptrend Anomalies:   {summary['up_anomalies']:3d}")
print(f"  Total Anomalies:            {summary['total_anomalies']:3d} ({summary['total_anomalies']/summary['total_days']*100:.1f}%)")

## 5. Visualization: Price with SMAs and Trend Cloud

In [ ]:
# Create trend fill bounds
df['SMA5_Green'] = np.where(df['SMA5'] >= df['SMA20'], df['SMA5'], df['SMA20'])
df['SMA5_Red'] = np.where(df['SMA5'] < df['SMA20'], df['SMA5'], df['SMA20'])

# Create interactive plot
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(f'Stock {STOCK_NAME} - Price with SMA Trend Cloud', 'Volume'),
    row_heights=[0.7, 0.3]
)

# Row 1: Trend Cloud & Price
# Base SMA20 (invisible)
fig.add_trace(go.Scatter(x=df['date'], y=df['SMA20'], line=dict(width=0), showlegend=False, hoverinfo='skip'), row=1, col=1)

# Green fill (uptrend)
fig.add_trace(go.Scatter(
    x=df['date'], y=df['SMA5_Green'],
    line=dict(width=0),
    fill='tonexty', fillcolor='rgba(0, 200, 0, 0.2)',
    name='Up Trend', hoverinfo='skip'
), row=1, col=1)

# Base again
fig.add_trace(go.Scatter(x=df['date'], y=df['SMA20'], line=dict(width=0), showlegend=False, hoverinfo='skip'), row=1, col=1)

# Red fill (downtrend)
fig.add_trace(go.Scatter(
    x=df['date'], y=df['SMA5_Red'],
    line=dict(width=0),
    fill='tonexty', fillcolor='rgba(200, 0, 0, 0.2)',
    name='Down Trend', hoverinfo='skip'
), row=1, col=1)

# SMA lines
fig.add_trace(go.Scatter(x=df['date'], y=df['SMA20'], line=dict(color='red', width=1.5), name='SMA 20'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=df['SMA5'], line=dict(color='orange', width=1.5), name='SMA 5'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=df['price'], line=dict(color='blue', width=1), name='Close Price'), row=1, col=1)

# Row 2: Volume
fig.add_trace(go.Bar(x=df['date'], y=df['volume'], marker_color='gray', name='Volume'), row=2, col=1)

fig.update_layout(
    title=f'Interactive Analysis: Stock {STOCK_NAME} (Price Trend)',
    xaxis_rangeslider_visible=False,
    height=700,
    template='plotly_white'
)

fig.show()

## 6. Visualization: SMA Gap Momentum Signals

In [ ]:
# Define signal colors
signal_colors = {
    'Strong Downtrend': 'red',
    'Signal Uptrend': 'green',
    'Strong Uptrend': 'darkgreen',
    'Signal Downtrend': 'orange',
    'Unknown': 'lightgray'
}

df['Signal_Color'] = df['Strategy_Signal'].map(signal_colors)

# Create plot
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(f'Stock {STOCK_NAME} - Price & SMAs', 'SMA Gap Momentum (Strategy Signal)'),
    row_heights=[0.6, 0.4]
)

# Row 1: Price & SMAs
fig.add_trace(go.Scatter(x=df['date'], y=df['price'], line=dict(color='blue', width=1), name='Close Price'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=df['SMA5'], line=dict(color='orange', width=1), name='SMA 5'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=df['SMA20'], line=dict(color='red', width=1), name='SMA 20'), row=1, col=1)

# Row 2: Gap Bar Chart with color-coded signals
fig.add_trace(go.Bar(
    x=df['date'], y=df['SMA_Diff'],
    marker_color=df['Signal_Color'],
    name='SMA Gap'
), row=2, col=1)

# Add dummy traces for legend
for signal, color in signal_colors.items():
    if signal != 'Unknown':
        fig.add_trace(go.Bar(x=[None], y=[None], marker_color=color, name=signal), row=2, col=1)

fig.update_layout(
    title=f'Strategy Analysis: SMA Gap Momentum for {STOCK_NAME}',
    xaxis_rangeslider_visible=False,
    height=800,
    template='plotly_white',
    barmode='overlay'
)

fig.show()

## 7. Visualization: Anomaly Detection

In [ ]:
# Create anomaly visualization
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(f'Stock {STOCK_NAME} - Strategy Anomalies', 'SMA Gap Momentum'),
    row_heights=[0.7, 0.3]
)

# Row 1: Price & Anomalies
fig.add_trace(go.Scatter(x=df['date'], y=df['price'], line=dict(color='blue', width=1), name='Close Price'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=df['SMA5'], line=dict(color='orange', width=1), name='SMA 5'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=df['SMA20'], line=dict(color='red', width=1), name='SMA 20'), row=1, col=1)

# Uptrend anomalies (price dropped when signal predicted up)
uptrend_anomalies = df[df['Strat_Up_Anomaly'] | df['Strat_SignalUp_Anomaly']]
fig.add_trace(go.Scatter(
    x=uptrend_anomalies['date'], y=uptrend_anomalies['price'],
    mode='markers', marker=dict(color='black', symbol='x', size=7),
    name='Uptrend Anomaly (Next Day Drop)'
), row=1, col=1)

# Downtrend anomalies (price rose when signal predicted down)
downtrend_anomalies = df[df['Strat_Down_Anomaly'] | df['Strat_SignalDown_Anomaly']]
fig.add_trace(go.Scatter(
    x=downtrend_anomalies['date'], y=downtrend_anomalies['price'],
    mode='markers', marker=dict(color='purple', symbol='circle-open', size=7),
    name='Downtrend Anomaly (Next Day Rise)'
), row=1, col=1)

# Row 2: Gap Bar Chart (context)
fig.add_trace(go.Bar(
    x=df['date'], y=df['SMA_Diff'],
    marker_color=df['Signal_Color'],
    name='SMA Gap'
), row=2, col=1)

fig.update_layout(
    title=f'Strategy Anomalies: {STOCK_NAME} (Gap Momentum)',
    xaxis_rangeslider_visible=False,
    height=800,
    template='plotly_white'
)

fig.show()

## 8. Generate Predictions

In [ ]:
# Generate predictions
predictions = predict_returns(P, V)

print(f"Generated {len(predictions)} predictions")
print(f"\nPrediction Statistics:")
print(f"  Mean:   {np.mean(predictions[1:]):.6f}")
print(f"  Median: {np.median(predictions[1:]):.6f}")
print(f"  Min:    {np.min(predictions[1:]):.6f}")
print(f"  Max:    {np.max(predictions[1:]):.6f}")
print(f"  Std:    {np.std(predictions[1:]):.6f}")

## 9. Calculate Actual Returns

In [ ]:
# Calculate actual returns
def target(P, V):
    n, Q = len(P), [0]
    for i in range(1, n):
        Q.append(P[i] / P[i - 1] - 1)
    return Q

actual_returns = target(P, V)

print(f"Actual Returns Statistics:")
print(f"  Mean:   {np.mean(actual_returns[1:]):.6f}")
print(f"  Median: {np.median(actual_returns[1:]):.6f}")
print(f"  Min:    {np.min(actual_returns[1:]):.6f}")
print(f"  Max:    {np.max(actual_returns[1:]):.6f}")
print(f"  Std:    {np.std(actual_returns[1:]):.6f}")

## 10. Evaluate Performance

In [ ]:
# Evaluation function
def evaluate(p, t, display=False):
    p, t = p[1:], t[1:]
    n, e, f = len(t), [], []
    
    for i in range(1, n):
        e.append(t[i] - p[i - 1])
        f.append(t[i])
    
    den = np.nanquantile(np.abs(e), 0.5) + 0.5 * np.nanquantile(np.abs(e), 0.9)
    num = np.nanquantile(np.abs(f), 0.5) + 0.5 * np.nanquantile(np.abs(f), 0.9)
    
    if display:
        print(f"\n\tbase = {round(num, 3)}  |  abs = {round(den, 3)}  |  rel = {round(1 - den / num, 3)}\n")
        plt.figure(figsize=(10, 6))
        plt.hist(e, bins=30, edgecolor='black', alpha=0.7)
        plt.xlabel('Prediction Error')
        plt.ylabel('Frequency')
        plt.title('Distribution of Prediction Errors')
        plt.grid(True, alpha=0.3)
        plt.show()
    
    return den, 1 - den / num

# Evaluate
abs_error, rel_score = evaluate(predictions, actual_returns, display=True)

print("="*70)
print(f"PERFORMANCE SUMMARY: {STOCK_NAME}")
print("="*70)
print(f"  Absolute Error: {abs_error:.6f} {'✓ PASS' if abs_error < 0.005 else '✗ FAIL'} (target: < 0.005)")
print(f"  Relative Score: {rel_score:.6f} {'✓ PASS' if rel_score > 0 else '✗ FAIL'} (target: > 0)")
print("="*70)

## 11. Visualization: Predicted vs Actual Returns

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    'date': df['date'],
    'predicted': predictions,
    'actual': actual_returns
})

# Create interactive plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=comparison_df['date'],
    y=comparison_df['actual'],
    mode='lines',
    name='Actual Returns',
    line=dict(color='blue', width=1)
))

fig.add_trace(go.Scatter(
    x=comparison_df['date'],
    y=comparison_df['predicted'],
    mode='lines',
    name='Predicted Returns',
    line=dict(color='red', width=1, dash='dot')
))

# Add zero line
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

fig.update_layout(
    title=f'Predicted vs Actual Returns: {STOCK_NAME}',
    xaxis_title='Date',
    yaxis_title='Return',
    height=600,
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

## 12. Visualization: Prediction Scatter Plot

In [ ]:
# Scatter plot: Predicted vs Actual
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=comparison_df['predicted'][1:],
    y=comparison_df['actual'][1:],
    mode='markers',
    marker=dict(size=5, opacity=0.6, color='blue'),
    name='Predictions'
))

# Add perfect prediction line (y=x)
min_val = min(comparison_df['predicted'].min(), comparison_df['actual'].min())
max_val = max(comparison_df['predicted'].max(), comparison_df['actual'].max())
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    line=dict(color='red', dash='dash'),
    name='Perfect Prediction'
))

fig.update_layout(
    title=f'Prediction Accuracy: {STOCK_NAME}',
    xaxis_title='Predicted Return',
    yaxis_title='Actual Return',
    height=600,
    template='plotly_white',
    showlegend=True
)

fig.update_xaxes(scaleanchor="y", scaleratio=1)
fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

## 13. Directional Accuracy Analysis

In [ ]:
# Calculate directional accuracy
pred_direction = np.sign(comparison_df['predicted'][1:])
actual_direction = np.sign(comparison_df['actual'][1:])

directional_accuracy = np.mean(pred_direction == actual_direction)

print("="*70)
print("DIRECTIONAL ACCURACY ANALYSIS")
print("="*70)
print(f"Directional Accuracy: {directional_accuracy*100:.2f}%")
print(f"\nBreakdown:")
print(f"  Correct Up predictions:   {np.sum((pred_direction > 0) & (actual_direction > 0))}")
print(f"  Correct Down predictions: {np.sum((pred_direction < 0) & (actual_direction < 0))}")
print(f"  Incorrect predictions:    {np.sum(pred_direction != actual_direction)}")
print(f"  Total predictions:        {len(pred_direction)}")

# Confusion matrix
up_predicted_up_actual = np.sum((pred_direction > 0) & (actual_direction > 0))
up_predicted_down_actual = np.sum((pred_direction > 0) & (actual_direction < 0))
down_predicted_up_actual = np.sum((pred_direction < 0) & (actual_direction > 0))
down_predicted_down_actual = np.sum((pred_direction < 0) & (actual_direction < 0))

print(f"\nConfusion Matrix:")
print(f"                    Actual Up    Actual Down")
print(f"  Predicted Up      {up_predicted_up_actual:6d}       {up_predicted_down_actual:6d}")
print(f"  Predicted Down    {down_predicted_up_actual:6d}       {down_predicted_down_actual:6d}")

## 14. Final Summary

In [ ]:
print("="*70)
print(f"FINAL SUMMARY: Stock {STOCK_NAME}")
print("="*70)
print(f"\n1. Data Statistics:")
print(f"   Trading Days: {len(P)}")
print(f"   Price Range: {P.min():.2f} - {P.max():.2f}")

print(f"\n2. Strategy Performance:")
print(f"   Total Anomalies: {summary['total_anomalies']} ({summary['total_anomalies']/summary['total_days']*100:.1f}%)")
print(f"   Strong Downtrend Anomaly Rate: {summary['down_anomalies']/summary['strong_downtrend_days']*100:.1f}%" if summary['strong_downtrend_days'] > 0 else "   Strong Downtrend Anomaly Rate: N/A")
print(f"   Strong Uptrend Anomaly Rate: {summary['up_anomalies']/summary['strong_uptrend_days']*100:.1f}%" if summary['strong_uptrend_days'] > 0 else "   Strong Uptrend Anomaly Rate: N/A")

print(f"\n3. Prediction Performance:")
print(f"   Absolute Error: {abs_error:.6f} {'✓ PASS' if abs_error < 0.005 else '✗ FAIL'}")
print(f"   Relative Score: {rel_score:.6f} {'✓ PASS' if rel_score > 0 else '✗ FAIL'}")
print(f"   Directional Accuracy: {directional_accuracy*100:.2f}%")

print(f"\n4. Recommendations:")
if abs_error >= 0.005:
    print(f"   - Absolute error is above target. Consider adding more features.")
if rel_score <= 0:
    print(f"   - Relative score is below target. Strategy needs refinement.")
if summary['total_anomalies'] / summary['total_days'] > 0.4:
    print(f"   - High anomaly rate ({summary['total_anomalies']/summary['total_days']*100:.1f}%). Signals are not strongly predictive.")
if directional_accuracy < 0.55:
    print(f"   - Low directional accuracy. Consider ensemble methods or additional indicators.")

print("\n" + "="*70)